In [ ]:
import os, sys
import numpy as np
import pandas as pd
from google.colab import drive
DRIVE = "/content/drive/MyDrive/volatility-forecast"
drive.mount("/content/drive", force_remount=True)
os.chdir(DRIVE)

# --- load locked work frame (same 3831-row trim as 05) ---
df = pd.read_csv("data/processed/dataset.csv", index_col=0, parse_dates=True)
work = df.dropna().copy()
assert len(work) == 3831, f"expected 3831, got {len(work)}"   # must match the 03-05 work frame

MIN_TRAIN = 1008   # stream starts exactly at f00 test start -> aligns to fold date map

# --- static HAR: fit ONCE on the first MIN_TRAIN rows, then never retrain ---
feat = ["rv1", "rv5", "rv21"]
Xall = work[feat].values
y    = work["y_rv1"].values                                     # h1 target (1-day-ahead RV)

Xtr  = np.column_stack([np.ones(MIN_TRAIN), Xall[:MIN_TRAIN]])
beta, *_ = np.linalg.lstsq(Xtr, y[:MIN_TRAIN], rcond=None)       # OLS, train-only

# forward predict on deployment span [MIN_TRAIN:], no refit
Xte      = np.column_stack([np.ones(len(work) - MIN_TRAIN), Xall[MIN_TRAIN:]])
har_pred = Xte @ beta

# --- h1 errors on deployment span ---
y_dep     = y[MIN_TRAIN:]
rv1_dep   = work["rv1"].values[MIN_TRAIN:]                       # persistence forecast = current rv1
e_model   = np.abs(y_dep - har_pred)
e_persist = np.abs(y_dep - rv1_dep)

# --- relativized streams (vol component divides out) ---
L = (e_model > e_persist).astype(float)                          # DDM:  1 = model lost that day
u = e_model / (e_model + e_persist + 1e-12)                      # ADWIN: 0.5 parity, ->1 model worse

dates_dep = work.index[MIN_TRAIN:]                               # date index aligned to streams

print("deployment span :", dates_dep[0].date(), "->", dates_dep[-1].date(), "| n =", len(L))
print("overall L mean  :", round(float(L.mean()), 3), "(model loss rate vs persistence)")
print("overall u mean  :", round(float(u.mean()), 3), "(0.5 = parity)")

# REBUILD streams on h21 — the horizon where ground-truth drift actually lives
y21 = work["y_rv21"].values                                  # 21-day-ahead RV target

Xtr21 = np.column_stack([np.ones(MIN_TRAIN), Xall[:MIN_TRAIN]])
beta21, *_ = np.linalg.lstsq(Xtr21, y21[:MIN_TRAIN], rcond=None)
har21 = np.column_stack([np.ones(len(work) - MIN_TRAIN), Xall[MIN_TRAIN:]]) @ beta21

y21_dep = y21[MIN_TRAIN:]
rv21_dep = work["rv21"].values[MIN_TRAIN:]                    # persistence(h21) = current rv21
e_model21   = np.abs(y21_dep - har21)
e_persist21 = np.abs(y21_dep - rv21_dep)

L21 = (e_model21 > e_persist21).astype(float)
u21 = e_model21 / (e_model21 + e_persist21 + 1e-12)

Mounted at /content/drive
deployment span : 2015-02-05 -> 2026-04-28 | n = 2823
overall L mean  : 0.383 (model loss rate vs persistence)
overall u mean  : 0.458 (0.5 = parity)


In [ ]:
from src.splits import walk_forward_splits

folds = list(walk_forward_splits(len(work)))
print("n folds :", len(folds))
for i, (tr, te) in enumerate(folds):
    print(f"f{i:02d}  test {work.index[te[0]].date()} -> {work.index[te[-1]].date()}")

n folds : 11
f00  test 2015-02-05 -> 2016-02-04
f01  test 2016-02-05 -> 2017-02-03
f02  test 2017-02-06 -> 2018-02-05
f03  test 2018-02-06 -> 2019-02-06
f04  test 2019-02-07 -> 2020-02-06
f05  test 2020-02-07 -> 2021-02-05
f06  test 2021-02-08 -> 2022-02-04
f07  test 2022-02-07 -> 2023-02-07
f08  test 2023-02-08 -> 2024-02-08
f09  test 2024-02-09 -> 2025-02-11
f10  test 2025-02-12 -> 2026-02-12


In [ ]:
# Fold-wise mean of the h21 streams (built in the first cell).
# Stream index 0 == work row MIN_TRAIN == f00 test start, so fold test
# indices shift by -MIN_TRAIN.
print("fold | window                    | L21    | u21    | note")
print("-" * 70)
for i, (tr, te) in enumerate(folds):
    s0, s1 = te[0] - MIN_TRAIN, te[-1] - MIN_TRAIN + 1
    if s0 < 0:
        continue
    gt = {5: "<- COVID  (NEG)", 7: "<- 2022   (POS, TP)", 8: "<- aftermath (NEG)"}.get(i, "")
    print(f"f{i:02d}  | {work.index[te[0]].date()} -> {work.index[te[-1]].date()} | "
          f"{L21[s0:s1].mean():.3f}  | {u21[s0:s1].mean():.3f}  | {gt}")

fold | window                    | L21    | u21    | note
----------------------------------------------------------------------
f00  | 2015-02-05 -> 2016-02-04 | 0.417  | 0.453  | 
f01  | 2016-02-05 -> 2017-02-03 | 0.468  | 0.488  | 
f02  | 2017-02-06 -> 2018-02-05 | 0.536  | 0.534  | 
f03  | 2018-02-06 -> 2019-02-06 | 0.365  | 0.452  | 
f04  | 2019-02-07 -> 2020-02-06 | 0.274  | 0.449  | 
f05  | 2020-02-07 -> 2021-02-05 | 0.421  | 0.434  | <- COVID  (NEG)
f06  | 2021-02-08 -> 2022-02-04 | 0.500  | 0.531  | 
f07  | 2022-02-07 -> 2023-02-07 | 0.694  | 0.598  | <- 2022   (POS, TP)
f08  | 2023-02-08 -> 2024-02-08 | 0.361  | 0.430  | <- aftermath (NEG)
f09  | 2024-02-09 -> 2025-02-11 | 0.294  | 0.438  | 
f10  | 2025-02-12 -> 2026-02-12 | 0.429  | 0.444  | 


In [ ]:
STRIDE = 5

# subsample streams; keep a parallel date array so fire indices map back to calendar
sub_idx   = np.arange(0, len(L21), STRIDE)          # positions in deployment-span coords
L21_s     = L21[sub_idx]
u21_s     = u21[sub_idx]
dates_s   = dates_dep[sub_idx]                       # date at each subsampled point
print("subsampled n :", len(u21_s), "| fold approx pts:", 252 // STRIDE)


from src.drift import adwin, ddm

r = adwin(u21_s, delta=0.002, min_n=20)
fire_pos = r["drift"]                                # indices into the SUBSAMPLED stream
print("\nADWIN fire count :", len(fire_pos))
print("ADWIN fire dates :", [dates_s[p].date().isoformat() for p in fire_pos])

# --- map each fire to the fold whose test window contains that date ---
def date_to_fold(d):
    for i, (tr, te) in enumerate(folds):
        if work.index[te[0]] <= d <= work.index[te[-1]]:
            return i
    return None

print("\nfire -> fold:")
for p in fire_pos:
    d = dates_s[p]
    print(f"  {d.date()}  -> f{date_to_fold(d)}")

subsampled n : 565 | fold approx pts: 50

ADWIN fire count : 0
ADWIN fire dates : []

fire -> fold:


In [ ]:
for d in [0.002, 0.01, 0.05, 0.1, 0.2, 0.3]:
    fires = adwin(u21_s, delta=d, min_n=20)["drift"]
    rows = [(dates_s[p].date().isoformat(), date_to_fold(dates_s[p])) for p in fires]
    print(f"delta={d:<5} n={len(fires):2d}  {rows}")

delta=0.002 n= 0  []
delta=0.01  n= 0  []
delta=0.05  n= 0  []
delta=0.1   n= 0  []
delta=0.2   n= 0  []
delta=0.3   n= 0  []


In [ ]:
# fallback / cross-check: DDM on binary L21 subsample
rb = ddm(L21_s, min_n=30)   # lower min_n: the subsampled stream has only n=565 points
print("DDM warnings:", [(dates_s[p].date().isoformat(), date_to_fold(dates_s[p])) for p in rb["warning"]])
print("DDM drifts  :", [(dates_s[p].date().isoformat(), date_to_fold(dates_s[p])) for p in rb["drift"]])

DDM warnings: [('2015-12-18', 0), ('2015-12-28', 0), ('2016-01-05', 0), ('2016-01-12', 0), ('2016-01-20', 0), ('2016-01-27', 0), ('2016-02-03', 0), ('2016-02-10', 1), ('2016-02-18', 1), ('2016-02-25', 1), ('2016-05-20', 1), ('2016-05-27', 1), ('2016-06-06', 1), ('2016-06-13', 1), ('2016-06-27', 1), ('2017-01-24', 1), ('2017-01-31', 1), ('2017-02-07', 2), ('2017-02-14', 2), ('2017-02-22', 2), ('2017-03-01', 2), ('2017-03-08', 2), ('2017-03-15', 2), ('2017-03-22', 2), ('2017-03-29', 2), ('2017-04-05', 2), ('2017-04-12', 2), ('2017-04-20', 2), ('2017-04-27', 2), ('2017-05-04', 2), ('2017-05-11', 2), ('2017-05-18', 2), ('2017-05-25', 2), ('2017-06-02', 2), ('2017-06-09', 2), ('2017-06-16', 2), ('2017-06-23', 2), ('2017-06-30', 2), ('2017-07-10', 2), ('2017-07-17', 2), ('2017-07-24', 2), ('2017-07-31', 2), ('2017-08-07', 2), ('2017-08-14', 2), ('2017-08-21', 2), ('2017-08-28', 2), ('2017-09-05', 2), ('2017-09-12', 2), ('2017-09-19', 2), ('2017-09-26', 2), ('2017-10-03', 2), ('2017-10-10', 2

In [ ]:
import numpy as np
from scipy import stats

OVERLAP = 21                       # h21 label horizon -> autocorrelation length
CALM = [0, 1, 3, 4, 6, 9, 10]      # f02 excluded (borderline), f05/f08 are TEST not baseline

# per-fold loss rate + effective n on the L21 stream (deployment coords)
def fold_slice(i):
    tr, te = folds[i]
    s0, s1 = te[0] - MIN_TRAIN, te[-1] - MIN_TRAIN + 1
    return L21[s0:s1]

# calm baseline p0 pooled over calm folds, with overlap-deflated n
calm_vals = np.concatenate([fold_slice(i) for i in CALM])
p0   = calm_vals.mean()
n0   = len(calm_vals) / OVERLAP
print(f"calm baseline p0 = {p0:.3f}  (n0_eff = {n0:.1f}, raw {len(calm_vals)})\n")

print("fold | p_hat | n_eff | z     | p_value | verdict        | truth")
print("-" * 72)
truth = {5: "NEG (COVID)", 7: "POS (2022)", 8: "NEG (aftermath)", 2: "borderline"}
for i in range(11):
    v = fold_slice(i)
    p_hat = v.mean()
    n_f   = len(v) / OVERLAP
    se    = np.sqrt(p_hat*(1-p_hat)/n_f + p0*(1-p0)/n0)
    z     = (p_hat - p0) / se if se > 0 else 0.0
    pval  = 1 - stats.norm.cdf(z)           # one-sided H1: p_f > p0
    flag  = "*** DRIFT" if pval < 0.05 else ("?  warn" if pval < 0.10 else "   ok")
    print(f"f{i:02d}  | {p_hat:.3f} | {n_f:4.1f}  | {z:+5.2f} | {pval:.4f}  | {flag:14s} | {truth.get(i,'')}")

calm baseline p0 = 0.392  (n0_eff = 84.0, raw 1764)

fold | p_hat | n_eff | z     | p_value | verdict        | truth
------------------------------------------------------------------------
f00  | 0.417 | 12.0  | +0.16 | 0.4363  |    ok          | 
f01  | 0.468 | 12.0  | +0.49 | 0.3104  |    ok          | 
f02  | 0.536 | 12.0  | +0.93 | 0.1751  |    ok          | borderline
f03  | 0.365 | 12.0  | -0.18 | 0.5725  |    ok          | 
f04  | 0.274 | 12.0  | -0.85 | 0.8025  |    ok          | 
f05  | 0.421 | 12.0  | +0.19 | 0.4261  |    ok          | NEG (COVID)
f06  | 0.500 | 12.0  | +0.70 | 0.2419  |    ok          | 
f07  | 0.694 | 12.0  | +2.11 | 0.0175  | *** DRIFT      | POS (2022)
f08  | 0.361 | 12.0  | -0.21 | 0.5831  |    ok          | NEG (aftermath)
f09  | 0.294 | 12.0  | -0.70 | 0.7566  |    ok          | 
f10  | 0.429 | 12.0  | +0.24 | 0.4060  |    ok          | 


In [ ]:
import pandas as pd

rows = []
for i in range(11):
    v = fold_slice(i)
    p_hat = v.mean()
    n_f = len(v) / OVERLAP
    se = np.sqrt(p_hat*(1-p_hat)/n_f + p0*(1-p0)/n0)
    z = (p_hat - p0) / se if se > 0 else 0.0
    pval = 1 - stats.norm.cdf(z)
    rows.append({
        "fold": i,
        "test_start": work.index[folds[i][1][0]].date().isoformat(),
        "test_end":   work.index[folds[i][1][-1]].date().isoformat(),
        "loss_rate":  round(p_hat, 4),
        "n_eff":      round(n_f, 1),
        "z":          round(z, 4),
        "p_value":    round(pval, 4),
        "drift":      bool(pval < 0.05),
    })

out = pd.DataFrame(rows)
out.to_csv("data/processed/concept_drift.csv", index=False)
print(out.to_string(index=False))
print(f"\nbaseline p0={p0:.4f} n0_eff={n0:.1f} | overlap={OVERLAP} | calm folds={CALM}")

 fold test_start   test_end  loss_rate  n_eff       z  p_value  drift
    0 2015-02-05 2016-02-04     0.4167   12.0  0.1604   0.4363  False
    1 2016-02-05 2017-02-03     0.4683   12.0  0.4946   0.3104  False
    2 2017-02-06 2018-02-05     0.5357   12.0  0.9343   0.1751  False
    3 2018-02-06 2019-02-06     0.3651   12.0 -0.1828   0.5725  False
    4 2019-02-07 2020-02-06     0.2738   12.0 -0.8505   0.8025  False
    5 2020-02-07 2021-02-05     0.4206   12.0  0.1863   0.4261  False
    6 2021-02-08 2022-02-04     0.5000   12.0  0.7001   0.2419  False
    7 2022-02-07 2023-02-07     0.6944   12.0  2.1093   0.0175   True
    8 2023-02-08 2024-02-08     0.3611   12.0 -0.2099   0.5831  False
    9 2024-02-09 2025-02-11     0.2937   12.0 -0.6953   0.7566  False
   10 2025-02-12 2026-02-12     0.4286   12.0  0.2380   0.4060  False

baseline p0=0.3923 n0_eff=84.0 | overlap=21 | calm folds=[0, 1, 3, 4, 6, 9, 10]
